In [6]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.utils
import time
import pickle
import random

# ------------------------------
# Global settings
# ------------------------------
SEED = 0
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

EPOCHS = 100
BATCH_SIZE = 1
KFOLD_SPLITS = 4
REPEATS = 300
OUTPUT_FILENAME = 'T18_combined.pkl'
CSV_PATH = '../CSN_Viability_Database_2025.csv'

# ------------------------------
# 1. Load and preprocess data
# ------------------------------
def load_CSN_data():
    return pd.read_csv(CSV_PATH)

# Load raw data
CSN = load_CSN_data()

# Drop unused columns
CSN = CSN.drop([
    'Example ID', 'Source', 'Figure ID', 'Data Provider', 'PI',
    'Date Received', 'Data Measurment Published',
    'Prior Exposure', 'Comments', 'Error'
], axis=1)

# One-hot encode categorical variables
CSN_prepared = pd.get_dummies(CSN, dtype=int)

# Feature engineering
CSN_prepared['Surface Area per Liter'] = (
    CSN_prepared['Surface Area (NMC) (m2/g)'] *
    CSN_prepared['Concentration (mg/L)']
)
CSN_prepared = CSN_prepared.drop(['Surface Area (NMC) (m2/g)'], axis=1)

CSN_prepared['log Concentration'] = np.log10(
    CSN_prepared['Concentration (mg/L)'] + 1e-9
)
CSN_prepared = CSN_prepared.drop(['Concentration (mg/L)'], axis=1)

# Hold-out test set: last 38 samples
CSN_test = CSN_prepared.tail(38)
CSN_train = CSN_prepared.iloc[:-38]

X_test = CSN_test.drop(['Viability_Fraction'], axis=1)
y_test = CSN_test['Viability_Fraction'].values

# ------------------------------
# 2. Define ANN model
# ------------------------------
def build_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(5, activation='relu'),
        layers.Dense(5, activation='relu'),
        layers.Dense(5, activation='relu'),
        layers.Dense(1, activation='relu')  # Output non-negative
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss='mse',
        metrics=['mae']
    )
    return model

# ------------------------------
# 3. Run K-Fold CV and bagging
# ------------------------------
def run_kfold_bagging(X, y, X_test, seeds, k=KFOLD_SPLITS):
    kf = KFold(n_splits=k, shuffle=True)

    fold_preds = np.zeros((k, X_test.shape[0]))
    weights = np.zeros(k)

    for i, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        tf.keras.backend.clear_session()
        seed = seeds[i]
        np.random.seed(seed)
        tf.random.set_seed(seed)

        model = build_model(X.shape[1])
        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )

        preds = model.predict(X_test).flatten()
        fold_preds[i] = preds
        weights[i] = 1.0 / (history.history['val_mae'][-1] + 1e-8)

    weighted_pred = np.average(fold_preds, axis=0, weights=weights)
    weighted_std_err = np.sqrt(np.average((fold_preds - weighted_pred) ** 2, axis=0, weights=weights)) / np.sqrt(k)

    return weighted_pred, weighted_std_err

# ------------------------------
# 4. Bagging 300 times
# ------------------------------
results = np.zeros((REPEATS, 2, X_test.shape[0]))  # [run, (mean, stderr), sample]

X_train_full = CSN_train.drop(['Viability_Fraction'], axis=1)
y_train_full = CSN_train['Viability_Fraction'].values

for run_idx in range(REPEATS):
    # Shuffle training set
    shuffled = sklearn.utils.shuffle(CSN_train, random_state=run_idx)
    X_train = shuffled.drop(['Viability_Fraction'], axis=1)
    y_train = shuffled['Viability_Fraction'].values

    # Fit scaler on training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Generate random seeds for folds
    seeds = np.random.randint(0, 100000, size=KFOLD_SPLITS)

    # Run KFold + prediction
    mean_pred, std_err = run_kfold_bagging(X_train_scaled, y_train, X_test_scaled, seeds)

    results[run_idx, 0] = mean_pred
    results[run_idx, 1] = std_err

    print(f"[{run_idx+1}/{REPEATS}] done.")

# ------------------------------
# 5. Save results
# ------------------------------
with open(OUTPUT_FILENAME, 'wb') as f:
    pickle.dump(results, f)

print(f"\n✅ Saved bagging results to: {OUTPUT_FILENAME}")


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[1/300] done.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
[2/300] done.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[3/300] done.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
[4/300] done.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
[5/300] done.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
[6/300] done.
2/2 

In [ ]:
# --------------------------
# MAE Statistics over REPEATS
# --------------------------
# Compute MAE per run
mae_list = []

for i in range(REPEATS):
    preds = results[i, 0]  # predicted mean
    mae = np.mean(np.abs(preds - y_test))
    mae_list.append(mae)

mae_array = np.array(mae_list)
mean_mae = np.mean(mae_array)
std_mae = np.std(mae_array)
min_mae = np.min(mae_array)
max_mae = np.max(mae_array)

# Print summary
print("\nSummary of Neural Network predictions over {} random seeds:".format(REPEATS))
print("Mean MAE: {:.4f} ± {:.4f}".format(mean_mae, std_mae))
print("Min MAE: {:.4f}".format(min_mae))
print("Max MAE: {:.4f}".format(max_mae))



Summary of Neural Network predictions over 300 random seeds:
Mean MAE: 0.4000 ± 0.0831
Min MAE: 0.2102
Max MAE: 0.7544
